<a href="https://colab.research.google.com/github/anusayannithar-ui/Volume_predictor/blob/main/Copy_of_Vo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers


In [ ]:
!pip install pillow
!pip install open3d
!pip install open3d.core


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.8/400.8 MB 10.6 MB/s  0:00:25
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 98.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 64.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 116.2 MB/s  0:00:00
  Attempting uninstall: widgetsnbextension
    Found existing installation: widgetsnbextension 3.6.10
    Uninstalling widgetsnbextension-3.6.10:
      Successfully uninstalled widgetsnbextension-3.6.10
  Attempting uninstall: ipywidgets
    Found existing installation: ipywidgets 7.7.1
    Uninstalling ipywidgets-7.7.1:
      Successfully uninstalled ipywidgets-7.7.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9/9 [open3d]
ERROR: Could not find a version that satisfies the requirement open3d.core (from versions: none)
ERROR: No matching distribution found for open3d.core


In [9]:
!pip install numpy
!pip install torch
!pip install open3d
!pip install math
!pip install pybind11



from PIL import Image
from transformers import pipeline
import open3d as o3d
import open3d.core as o3c
import numpy as np
import torch.utils.dlpack
import math as m




pipe = pipeline(task="depth-estimation",  model ="depth-anything/Depth-Anything-V2-Base-hf")


from PIL import Image


from transformers import pipeline


pipe = pipeline(task="depth-estimation",  model ="depth-anything/Depth-Anything-V2-Base-hf")


def DepthMapProcess(function):
     def wrapper():
          print("Creating depthmap")
          result = function()


          print("Depthmap is created")

          return result


     return wrapper


@DepthMapProcess
def createDepthmap():
  with Image.open("image.jpg") as img:
     return pipe(img)



depth_output = createDepthmap()
print(depth_output)


depthmap_points = depth_output['predicted_depth']

print(type(depthmap_points))

depth_3d = depthmap_points.unsqueeze(-1).float()


o3d_tensor = o3d.core.Tensor.from_dlpack(
    torch.utils.dlpack.to_dlpack(depth_3d.contiguous())
)

o3d_depth_image = o3d.t.geometry.Image(o3d_tensor)

height,width = depthmap_points.shape

hfov_degree = 75.0    # set cutom value .But need research (exif metadata)
hfov_rad = m.radians(hfov_degree)

calculated_foc_len = width/(2*m.tan(hfov_rad/2))




fx,fy=calculated_foc_len,calculated_foc_len
cx,cy = width/2.0,height/2.0

intrinsic_matrix = o3d.core.Tensor([[fx,0,cx],[0,fy,cy]],dtype=o3d.core.float64)


camera_model = o3d.pybind.camera.PinholeCameraIntrinsic(width,height,fx,fy,cx,cy)

pcd = o3d.t.geometry.PointCloud.create_from_depth_image(o3d_depth_image,camera_model)




#visuals = o3d.visulization.draw_geometries([pcd.to_legacy()],)
visuals = o3d.visulization.draw_plotly([pcd],)

with open("visuals.pcd",'w') as file:
  file.write(visuals)




























ERROR: Could not find a version that satisfies the requirement math (from versions: none)
ERROR: No matching distribution found for math


Loading weights:   0%|          | 0/287 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/287 [00:00<?, ?it/s]

Creating depthmap
Depthmap is created
{'predicted_depth': tensor([[2.8046, 2.8046, 2.8046,  ..., 2.3084, 2.3061, 2.3083],
        [2.8046, 2.8046, 2.8046,  ..., 2.3061, 2.3038, 2.3060],
        [2.8046, 2.8046, 2.8046,  ..., 2.3084, 2.3061, 2.3083],
        ...,
        [3.1133, 3.1137, 3.1132,  ..., 2.9918, 2.9921, 2.9918],
        [3.1138, 3.1142, 3.1138,  ..., 2.9919, 2.9922, 2.9919],
        [3.1133, 3.1137, 3.1132,  ..., 2.9918, 2.9921, 2.9918]]), 'depth': <PIL.Image.Image image mode=L size=4800x6000 at 0x7B637856BF50>}
<class 'torch.Tensor'>


TypeError: create_from_depth_image(): incompatible function arguments. The following argument types are supported:
    1. (depth: open3d.pybind.t.geometry.Image, intrinsics: open3d.pybind.core.Tensor, extrinsics: open3d.pybind.core.Tensor = [[1 0 0 0], [0 1 0 0], [0 0 1 0], [0 0 0 1]] Tensor[shape={4, 4}, stride={4, 1}, Float32, CPU:0, 0x350456f0], depth_scale: typing.SupportsFloat = 1000.0, depth_max: typing.SupportsFloat = 3.0, stride: typing.SupportsInt = 1, with_normals: bool = False) -> open3d.pybind.t.geometry.PointCloud

Invoked with: Image[size={6000,4800}, channels=1, Float32, CPU:0], PinholeCameraIntrinsic(width=4800, height=6000, )